# Concrete Crack Classification
### Custom CNN vs. Transfer Learning on the METU Concrete Crack Dataset

**Pipeline:**
1. Setup & data download
2. Data exploration & integrity checks
3. Stratified train/val/test split (patch-level; source-image grouping not available — see note)
4. Baseline: Custom CNN (trained from scratch)
5. Improved model: Transfer learning (EfficientNetB0, fine-tuned)
6. Evaluation: accuracy, precision, recall, F1, ROC-AUC, confusion matrix
7. Explainability: Grad-CAM
8. Model comparison & error analysis

> **Note on data leakage:** The public release of this dataset does not retain filenames that trace patches back to their 458 source high-resolution images. Ideally, a group-aware split (all patches from the same source image in the same fold) prevents the model from "cheating" via shared background texture. Since that grouping isn't recoverable here, we use a stratified random split instead and explicitly note this as a limitation of the results below.


## 1. Setup

In [ ]:
!pip install -q scikit-learn seaborn --upgrade


In [ ]:
import os
import random
import shutil
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, precision_recall_curve)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 2. Load the dataset

Before running this cell, attach the dataset via the right-hand panel:
click **Add Input** → search **"concrete-crack-images-for-classification"** (by arnavr10880) → attach it.

Also set your **Accelerator** (right-hand panel) to **GPU T4 x2** or **P100** before running any training cells below.


In [ ]:
# Kaggle can mount inputs at different depths depending on how they were added
# (sometimes /kaggle/input/<dataset>/, sometimes nested under /kaggle/input/datasets/<owner>/<dataset>/),
# so we search the whole tree for the Positive/Negative folders directly rather than assuming a fixed depth.

import pathlib

KAGGLE_INPUT_ROOT = pathlib.Path("/kaggle/input")

pos_matches = list(KAGGLE_INPUT_ROOT.rglob("Positive"))
neg_matches = list(KAGGLE_INPUT_ROOT.rglob("Negative"))

if pos_matches and neg_matches:
    POS_DIR = pos_matches[0]
    NEG_DIR = neg_matches[0]
    path = str(POS_DIR.parent)
    print("Found Positive dir:", POS_DIR)
    print("Found Negative dir:", NEG_DIR)
else:
    path = None
    POS_DIR = None
    NEG_DIR = None
    print("Could not find Positive/Negative folders automatically.")
    print("Make sure you've attached the dataset via 'Add Input' in the right-hand panel.")
    print("Full /kaggle/input tree below — locate the folders manually and set POS_DIR/NEG_DIR yourself if needed:")
    for p in KAGGLE_INPUT_ROOT.rglob("*"):
        if p.is_dir():
            print(p)


In [ ]:
# Inspect the folder structure so we know exactly what we're working with
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level >= 1:
        print(f'{indent}  ... ({len(files)} files)')


In [ ]:
# Confirm the paths (already set in the previous cell — this is just a sanity check)
print("Positive dir:", POS_DIR)
print("Negative dir:", NEG_DIR)


## 3. Data exploration & integrity checks

In [ ]:
pos_files = sorted(list(POS_DIR.glob("*.jpg")))
neg_files = sorted(list(NEG_DIR.glob("*.jpg")))

print(f"Positive (crack) images: {len(pos_files)}")
print(f"Negative (no crack) images: {len(neg_files)}")
print(f"Total: {len(pos_files) + len(neg_files)}")

# Sanity check on image size/mode as described in the dataset card (227x227 RGB)
from PIL import Image
sample = Image.open(pos_files[0])
print("Sample image size/mode:", sample.size, sample.mode)


In [ ]:
# Visualize a few samples from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    axes[0, i].imshow(Image.open(pos_files[i]))
    axes[0, i].set_title("Crack")
    axes[0, i].axis("off")
    axes[1, i].imshow(Image.open(neg_files[i]))
    axes[1, i].set_title("No crack")
    axes[1, i].axis("off")
plt.suptitle("Sample images")
plt.tight_layout()
plt.show()


In [ ]:
# Check for corrupted images (worth doing before training on 40k files)
def find_corrupted(file_list):
    bad = []
    for f in file_list:
        try:
            img = Image.open(f)
            img.verify()
        except Exception:
            bad.append(f)
    return bad

bad_pos = find_corrupted(pos_files)
bad_neg = find_corrupted(neg_files)
print(f"Corrupted positive files: {len(bad_pos)}")
print(f"Corrupted negative files: {len(bad_neg)}")

pos_files = [f for f in pos_files if f not in bad_pos]
neg_files = [f for f in neg_files if f not in bad_neg]


## 4. Train / Validation / Test split

We use a **70/15/15 stratified split**, keeping class balance identical across all three sets.


In [ ]:
df = pd.DataFrame({
    "filepath": [str(f) for f in pos_files] + [str(f) for f in neg_files],
    "label": [1] * len(pos_files) + [0] * len(neg_files)  # 1 = crack, 0 = no crack
})

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
print("\nClass balance (train):\n", train_df["label"].value_counts(normalize=True))


In [ ]:
IMG_SIZE = (227, 227)   # native resolution per dataset card — no resizing distortion
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def make_dataset(df, training=False, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((df["filepath"].values, df["label"].values))
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(buffer_size=2000, seed=SEED)
    if augment:
        ds = ds.map(lambda x, y: (augment_and_clip(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

# Mild augmentation — the dataset card notes NO augmentation was applied originally,
# so this also serves as an ablation point: does light augmentation help generalization?
#
# IMPORTANT: our images are normalized to [0, 1] (see load_and_preprocess above).
# RandomBrightness defaults to assuming a [0, 255] pixel range, which would corrupt
# already-normalized images. We explicitly set value_range=(0, 1) to match, and clip
# afterwards as a safety net since these layers can still push values slightly out of range.
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.05),
    layers.RandomBrightness(0.1, value_range=(0, 1)),
    layers.RandomContrast(0.1),
])

def augment_and_clip(x, training=True):
    x = data_augmentation(x, training=training)
    return tf.clip_by_value(x, 0.0, 1.0)

train_ds = make_dataset(train_df, training=True, augment=True)
val_ds   = make_dataset(val_df, training=False, augment=False)
test_ds  = make_dataset(test_df, training=False, augment=False)


## 5. Baseline model — Custom CNN from scratch

In [ ]:
def build_custom_cnn(input_shape=(227, 227, 3)):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(1, activation="sigmoid"),
    ], name="custom_cnn")
    return model

custom_cnn = build_custom_cnn()
custom_cnn.summary()


In [ ]:
custom_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.Precision(name="precision"),
             keras.metrics.Recall(name="recall"), keras.metrics.AUC(name="auc")]
)

callbacks_cnn = [
    keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint("custom_cnn_best.keras", monitor="val_auc", mode="max", save_best_only=True),
]

history_cnn = custom_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=callbacks_cnn
)


## 6. Improved model — Transfer learning (EfficientNetB0)

In [ ]:
def build_transfer_model(input_shape=(227, 227, 3), fine_tune_at=None):
    base_model = keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=input_shape
    )
    base_model.trainable = False  # freeze for initial training phase

    inputs = keras.Input(shape=input_shape)
    # EfficientNet expects 0-255 range internally via its own preprocessing layer
    x = layers.Rescaling(255.0)(inputs)  # undo our earlier /255 normalization
    x = keras.applications.efficientnet.preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs, outputs, name="efficientnet_transfer")
    return model, base_model

transfer_model, base_model = build_transfer_model()
transfer_model.summary()


In [ ]:
# Phase 1: train the new head only (base frozen)
transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.Precision(name="precision"),
             keras.metrics.Recall(name="recall"), keras.metrics.AUC(name="auc")]
)

callbacks_transfer = [
    keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("transfer_head_best.keras", monitor="val_auc", mode="max", save_best_only=True),
]

history_head = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks_transfer
)


In [ ]:
# Phase 2: unfreeze the top block(s) of the base model and fine-tune with a low LR
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30   # unfreeze last ~30 layers
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.Precision(name="precision"),
             keras.metrics.Recall(name="recall"), keras.metrics.AUC(name="auc")]
)

callbacks_finetune = [
    keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("transfer_finetuned_best.keras", monitor="val_auc", mode="max", save_best_only=True),
]

history_finetune = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_finetune
)


## 7. Training curves

In [ ]:
def plot_history(histories, labels, metric="auc"):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    for h, lbl in zip(histories, labels):
        plt.plot(h.history[metric], label=f"{lbl} - train")
        plt.plot(h.history[f"val_{metric}"], "--", label=f"{lbl} - val")
    plt.title(metric.upper())
    plt.xlabel("Epoch"); plt.legend()

    plt.subplot(1, 2, 2)
    for h, lbl in zip(histories, labels):
        plt.plot(h.history["loss"], label=f"{lbl} - train")
        plt.plot(h.history["val_loss"], "--", label=f"{lbl} - val")
    plt.title("Loss")
    plt.xlabel("Epoch"); plt.legend()
    plt.tight_layout()
    plt.show()

plot_history([history_cnn], ["Custom CNN"])
plot_history([history_finetune], ["EfficientNet (fine-tune phase)"])


## 8. Evaluation on the held-out test set

In [ ]:
def evaluate_model(model, test_ds, test_df, name):
    y_true = test_df["label"].values
    y_prob = model.predict(test_ds).flatten()
    y_pred = (y_prob >= 0.5).astype(int)

    print(f"\n=== {name} — Test set report ===")
    print(classification_report(y_true, y_pred, target_names=["No crack", "Crack"]))
    auc = roc_auc_score(y_true, y_prob)
    print(f"ROC-AUC: {auc:.4f}")

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No crack", "Crack"], yticklabels=["No crack", "Crack"])
    plt.title(f"{name} — Confusion Matrix")
    plt.ylabel("True"); plt.xlabel("Predicted")
    plt.show()

    return y_true, y_prob, auc

results = {}
results["Custom CNN"] = evaluate_model(custom_cnn, test_ds, test_df, "Custom CNN")
results["EfficientNetB0 (fine-tuned)"] = evaluate_model(transfer_model, test_ds, test_df, "EfficientNetB0 (fine-tuned)")


In [ ]:
# Combined ROC curve for direct comparison
plt.figure(figsize=(6, 6))
for name, (y_true, y_prob, auc) in results.items():
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.4f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()


## 9. Explainability — Grad-CAM

In [ ]:
def build_gradcam_ready_model(seq_model, last_conv_layer_name):
    """Sequential models (like our custom_cnn) don't expose `.output` for
    intermediate layers until they've been traced symbolically. This rebuilds
    the model as a functional graph using the SAME already-trained layers/weights
    — no retraining needed — so Grad-CAM can access the conv layer's activations."""
    inputs = keras.Input(shape=seq_model.input_shape[1:])
    x = inputs
    conv_output = None
    for layer in seq_model.layers:
        x = layer(x)
        if layer.name == last_conv_layer_name:
            conv_output = x
    if conv_output is None:
        raise ValueError(f"Layer '{last_conv_layer_name}' not found in model.")
    return keras.Model(inputs, [conv_output, x])


def make_gradcam_heatmap(img_array, grad_model, pred_index=None):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = 0
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def show_gradcam(grad_model, img_path, img_size=IMG_SIZE):
    img = Image.open(img_path).convert("RGB").resize(img_size)
    img_array = np.expand_dims(np.array(img) / 255.0, axis=0)

    heatmap = make_gradcam_heatmap(img_array, grad_model)
    heatmap = np.uint8(255 * heatmap)
    heatmap_img = Image.fromarray(heatmap).resize(img_size)
    heatmap_img = plt.cm.jet(np.array(heatmap_img) / 255.0)[:, :, :3]

    overlay = 0.6 * np.array(img) / 255.0 + 0.4 * heatmap_img

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img); axes[0].set_title("Original"); axes[0].axis("off")
    axes[1].imshow(overlay); axes[1].set_title("Grad-CAM"); axes[1].axis("off")
    plt.tight_layout()
    plt.show()

# Find the last Conv2D layer in custom_cnn
last_conv_name = [l.name for l in custom_cnn.layers if isinstance(l, layers.Conv2D)][-1]
print("Using layer:", last_conv_name)

# Rebuild as a traceable functional model (reuses trained weights, no retraining)
gradcam_model = build_gradcam_ready_model(custom_cnn, last_conv_name)

sample_crack_path = test_df[test_df["label"] == 1].iloc[0]["filepath"]
show_gradcam(gradcam_model, sample_crack_path)


In [ ]:
custom_cnn.save("custom_cnn_final.keras")
transfer_model.save("efficientnet_final.keras")
print("Models saved.")
